# NJ148 training on Colab (L4), v2

Run the cells in order, on a fresh server. Packages are pinned to the tested transformers 4.46.3 stack; transformers 5.x is not used.
Outputs show counts, hashes and pass/fail only, never note text.

Needs `MyDrive/nj148/nj148_transfer.tar.gz` (code bundle, gold_v2, synthetic data). Results are copied to `MyDrive/nj148/runs/` after each step.

In [ ]:
# 1. GPU and Drive
import torch, os, pathlib
assert torch.cuda.is_available(), 'no GPU: the server must be an L4'
print(torch.cuda.get_device_name(0), 'bf16', torch.cuda.is_bf16_supported())
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/nj148')
RUNS = DRIVE / 'runs'
for sub in ('checkpoints', 'predictions_secure', 'logs'):
    (RUNS / sub).mkdir(parents=True, exist_ok=True)
print('archive present:', (DRIVE / 'nj148_transfer.tar.gz').exists())

In [ ]:
# 2. Unpack, clone the committed code from the bundle, place gold and synthetic data
import subprocess, hashlib, json, shutil
WORK, REPO = pathlib.Path('/content/work'), pathlib.Path('/content/repo')
sh = lambda cmd: print(subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True).stdout.strip())
shutil.rmtree(WORK, ignore_errors=True); shutil.rmtree(REPO, ignore_errors=True); WORK.mkdir()
sh(f'tar -xzf {DRIVE}/nj148_transfer.tar.gz -C {WORK}')
manifest = json.loads((WORK / 'transfer_manifest.json').read_text())
sh(f'git clone -q -b research-reconstruction {WORK}/repo.bundle {REPO}')
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == manifest['commit'], f"commit mismatch {head} vs {manifest['commit']}"
(REPO / 'data_secure' / 'gold').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'gold_v2.jsonl', REPO / 'data_secure' / 'gold' / 'gold_v2.jsonl')
sha = hashlib.sha256((REPO / 'data_secure/gold/gold_v2.jsonl').read_bytes()).hexdigest()
assert sha == manifest['gold_sha256'], 'gold hash mismatch'
sh(f'cp -R {WORK}/synthetic_data/. {REPO}/synthetic_data/')
bad = [p for p, h in manifest['files'].items() if hashlib.sha256((REPO / p).read_bytes()).hexdigest() != h]
assert not bad, f'{len(bad)} files differ from the manifest'
# Official outputs must be real folders inside the repo: the code resolves paths, so a symlink to Drive would turn an
# official run into a development run. Results are copied to Drive after each step and restored from it here.
def backup():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {REPO}/{sub}/ {RUNS}/{sub}/')
    print('backed up to Drive')
def restore():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {RUNS}/{sub}/ {REPO}/{sub}/')
restore()
print('commit', head[:7], '| gold sha', sha[:8], '| files verified', len(manifest['files']))

In [ ]:
# 3. Packages pinned to the versions the pipeline was tested with (local smoke test). transformers 5.x is NOT used.
# torch stays Colab's (no torch 2.2.2 build exists for Python 3.13 / CUDA 13); bitsandbytes 0.50.2 passed NF4 on this L4.
PINS = {"transformers": "4.46.3", "peft": "0.21.1", "accelerate": "1.14.0", "tokenizers": "0.20.3", "safetensors": "0.8.0", "huggingface_hub": "0.36.2", "bitsandbytes": "0.50.2"}
spec = ' '.join(f'{k}=={v}' for k, v in PINS.items())
!pip -q install {spec} sentencepiece PyYAML scikit-learn pytest 2>&1 | tail -3
import importlib.metadata as md
got = {k: md.version(k) for k in PINS}
print(got, '| torch', md.version('torch'))
wrong = {k: (got[k], v) for k, v in PINS.items() if got[k] != v}
assert not wrong, f'version mismatch (installed, pinned): {wrong}'
!pip freeze > /content/drive/MyDrive/nj148/runs/logs/requirements-training.colab.lock
print('pins ok; lock written to Drive')

In [ ]:
!pip -q uninstall -y torchao
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'
print('torchao removed')


In [ ]:
# 3b. The same mechanical smoke test that passed locally: tiny random model, invented data, no project data, no download
import subprocess
r = subprocess.run(['python', 'scripts/smoke_tiny_pipeline.py'], cwd=REPO, capture_output=True, text=True)
print('exit', r.returncode)
print((r.stdout + r.stderr)[-3000:])

In [ ]:
# 4. Hugging Face login. The token is typed into the prompt and never stored in this notebook
from getpass import getpass
from huggingface_hub import login, whoami
login(token=getpass('Hugging Face token: '), add_to_git_credential=False)
print('logged in as', whoami()['name'])

In [ ]:
# 5. Preflight with the pinned model and the gold check. 'GPU is an RTX 3070' is expected to fail on an L4
import subprocess
out = RUNS / 'logs' / 'preflight_colab_l4.json'
r = subprocess.run(['python', 'scripts/preflight_rtx3070.py', '--out', str(out), '--model-id', 'meta-llama/Llama-3.2-3B-Instruct',
                    '--revision', '0cb88a4f764b7a12671c53f0838cd831a0843b95', '--check-gold'], cwd=REPO, capture_output=True, text=True)
print('exit', r.returncode)
print(r.stdout[-6000:])
if not out.exists():
    print(r.stderr[-2000:])

In [ ]:
# 6. Timed pilot: E1 fold 1 as a DEVELOPMENT run (not official, no held-out inference). Only measures time
import time, subprocess
t0 = time.time()
log = RUNS / 'logs' / 'pilot_E1_fold1.log'
with open(log, 'w') as f:
    r = subprocess.run(['python', '-m', 'src.training.cli', 'train', '--experiment', 'E1', '--fold', '1', '--seed', '42'],
                       cwd=REPO, stdout=f, stderr=subprocess.STDOUT, text=True)
print('exit', r.returncode, '| minutes', round((time.time() - t0) / 60, 1))
print(open(log).read()[-1500:])
backup()